# 📅 Day 17 — Mock Interview Practice

**Week 3 · Day 3 · Level: ADVANCED (pushing into HARD)**

Week 3 continues: **product-company level** questions — complex end-to-end scenarios, optimization trade-offs, and system-design aspects. 18 questions across 8 categories — 14 [ADVANCED] + 4 [HARD]. Today's themes: recursive CTEs for hierarchies, sliding-window algorithms, CDC processing, Tungsten codegen internals, dead-letter queues, Data Vault 2.0, Snowflake dynamic tables, and pipeline-vs-data monitoring. Every answer should come with a trade-off articulated out loud.

| # | Category | Questions | Level |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Q1-Q2 | ADVANCED |
| 2 | Python — Loop/Function/List/Dict | Q3-Q4 | ADVANCED |
| 3 | PySpark Coding | Q5-Q6 | ADVANCED |
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | ADVANCED / HARD |
| 5 | ETL — Fault Tolerance | Q9-Q11 | ADVANCED / HARD |
| 6 | Data Modeling | Q12-Q13 | ADVANCED / HARD |
| 7 | Snowflake | Q14-Q15 | ADVANCED |
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | ADVANCED |

**Score goal**: articulate every solution out loud within 4 minutes. Trade-offs must be named.

## Q1 [ADVANCED] SQL Advanced — Recursive CTE: Employee Hierarchy with Depth & Path

**Scenario**: A company stores employees with their manager in a flat table: `(emp_id, emp_name, manager_id, salary)`. The CEO has `manager_id = NULL`. The HR team needs a **full org chart** showing each employee's depth from the CEO, a breadcrumb path (CEO → VP → Director → …), and the total salary of their entire reporting subtree. This is a classic recursive CTE problem asked at Meta and Amazon for analytics engineering roles.

**Requirements**:
1. Write a **recursive CTE** that starts from the CEO (`manager_id IS NULL`) and walks down the management chain.
2. Track the **depth** (CEO = 0, direct reports = 1, etc.) and a **breadcrumb path** of names (e.g., `CEO → VP_Eng → Dir_Data → Eng_Alice`).
3. Compute the **total subtree salary** — the sum of salaries for the employee plus ALL their direct and indirect reports (recursive aggregation).
4. Return the full org chart ordered by depth, then name.

**Sample data**: CEO (salary 500K) → two VPs (Eng 300K, Sales 280K) → each VP has 2 reports. 7 employees total.

**Expected output**: 7 rows with depth, path, and subtree salary (CEO's subtree = all salaries).

In [0]:
# ── Solution Q1: recursive CTE for org chart with depth, path, subtree salary ───
from pyspark.sql.functions import col

employees = [
    (1, "CEO", None, 500000),
    (2, "VP_Eng", 1, 300000),
    (3, "VP_Sales", 1, 280000),
    (4, "Dir_Data", 2, 200000),
    (5, "Dir_Platform", 2, 190000),
    (6, "Mgr_AM", 3, 170000),
    (7, "Mgr_SE", 3, 160000),
]
schema = ["emp_id", "emp_name", "manager_id", "salary"]
emp_df = spark.createDataFrame(employees, schema)
emp_df.createOrReplaceTempView("employees")

result = spark.sql("""
WITH RECURSIVE org_base AS (
    -- Anchor: the CEO (depth 0, path = own name)
    SELECT emp_id, emp_name, manager_id, salary,
           0 AS depth,
           emp_name AS path,
           array(emp_id) AS subtree_ids
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    -- Recursive: join children to parents already in the CTE
    SELECT c.emp_id, c.emp_name, c.manager_id, c.salary,
           p.depth + 1 AS depth,
           concat_ws(' → ', p.path, c.emp_name) AS path,
           array_union(p.subtree_ids, array(c.emp_id)) AS subtree_ids
    FROM employees c
    INNER JOIN org_base p ON c.manager_id = p.emp_id
)
SELECT o.emp_id, o.emp_name, o.depth, o.path,
       (SELECT SUM(e.salary) FROM employees e
        WHERE array_contains(o.subtree_ids, e.emp_id)) AS subtree_salary
FROM org_base o
ORDER BY o.depth, o.emp_name
""")
result.show(truncate=False)

# Verify: CEO's subtree salary = sum of ALL salaries = 500K + 300K + 280K + 200K + 190K + 170K + 160K = 1,800,000
ceo_row = result.filter(col("emp_name") == "CEO").first()
assert ceo_row["subtree_salary"] == 1800000, f"CEO subtree should be 1.8M, got {ceo_row['subtree_salary']}"
print(f"Q1 OK — CEO subtree salary = {ceo_row['subtree_salary']:,} ✓ (sum of all 7 salaries)")

# Verify depth: CEO=0, VPs=1, Directors=2
vp_row = result.filter(col("emp_name") == "VP_Eng").first()
assert vp_row["depth"] == 1, f"VP_Eng depth should be 1, got {vp_row['depth']}"
dir_row = result.filter(col("emp_name") == "Dir_Data").first()
assert dir_row["depth"] == 2, f"Dir_Data depth should be 2, got {dir_row['depth']}"
print(f"Q1 OK — depths verified: CEO=0, VP_Eng={vp_row['depth']}, Dir_Data={dir_row['depth']}")
print(f"Q1 OK — path example: {dir_row['path']}")

## Q2 [ADVANCED] SQL Advanced — Time-Series Gap Filling & Forward Fill

**Scenario**: A sensor network reports temperature readings per station every hour. But sensors go offline — there are **gaps** in the data. The analytics team needs a **continuous time series** where every (station, hour) combination exists, and missing values are **forward-filled** from the last known reading. This is a common IoT analytics interview question at Uber and Google.

**Requirements**:
1. Generate a **complete hourly grid** for every station — every hour between the station's first and last reading must exist, even if no sensor data was recorded.
2. **Forward-fill** missing temperature values from the last known reading (a gap of 3 hours gets the last value before the gap).
3. If there is no prior reading (gap at the start), leave it as NULL (no backward fill).
4. Flag each row as `is_actual` (true if from a real reading, false if forward-filled).

**Sample data**: Station A reads at hours 1 (20°), 4 (22°), 7 (25°). Hours 2,3,5,6 are missing. Expected output: 7 rows (hours 1-7), with hour 2,3 filled with 20°, hours 5,6 filled with 22°.

**Expected output**: 7 rows with station, hour, temperature (forward-filled), is_actual flag.

In [0]:
# ── Solution Q2: time-series gap filling with forward fill ─────────────────────
from pyspark.sql import Window
from pyspark.sql.functions import col, explode, sequence, to_timestamp, min as spark_min, max as spark_max, lit, last, when

readings = [
    ("stationA", 1, 20.0),
    ("stationA", 4, 22.0),
    ("stationA", 7, 25.0),
    ("stationB", 1, 15.0),
    ("stationB", 3, 18.0),
    ("stationB", 5, 21.0),
]
rdf = spark.createDataFrame(readings, ["station", "hour", "temperature"])
rdf.createOrReplaceTempView("readings")

result = spark.sql("""
WITH bounds AS (
    -- Find the min and max hour per station
    SELECT station, min(hour) AS min_hr, max(hour) AS max_hr
    FROM readings
    GROUP BY station
),
hour_grid AS (
    -- Generate every hour between min and max for each station
    SELECT station, hour
    FROM bounds
    LATERAL VIEW explode(sequence(min_hr, max_hr)) t AS hour
),
joined AS (
    -- Left join real readings onto the complete grid
    SELECT g.station, g.hour, r.temperature,
           CASE WHEN r.temperature IS NOT NULL THEN true ELSE false END AS is_actual
    FROM hour_grid g
    LEFT JOIN readings r ON g.station = r.station AND g.hour = r.hour
)
SELECT station, hour,
       -- Forward fill: last non-null temperature, bounded by station partition
       last(temperature, true) OVER (
           PARTITION BY station ORDER BY hour
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS temperature,
       is_actual
FROM joined
ORDER BY station, hour
""")
result.show(truncate=False)

# Verify station A: hour 1=20.0 (actual), hour 2=20.0 (filled), hour 3=20.0 (filled),
#                  hour 4=22.0 (actual), hour 5=22.0 (filled), hour 6=22.0 (filled), hour 7=25.0 (actual)
stationA = result.filter(col("station") == "stationA").orderBy("hour").collect()
assert len(stationA) == 7, f"Station A should have 7 rows, got {len(stationA)}"
assert stationA[0]["temperature"] == 20.0 and stationA[0]["is_actual"] == True
assert stationA[1]["temperature"] == 20.0 and stationA[1]["is_actual"] == False, f"Hour 2 should be filled 20.0, got {stationA[1]}"
assert stationA[4]["temperature"] == 22.0 and stationA[4]["is_actual"] == False, f"Hour 5 should be filled 22.0, got {stationA[4]}"
assert stationA[6]["temperature"] == 25.0 and stationA[6]["is_actual"] == True
print("Q2 OK — 7 rows for stationA, forward fill verified (hours 2,3→20°, hours 5,6→22°)")

## Q3 [ADVANCED] Python — Sliding Window Maximum (Monotonic Deque)

**Scenario**: You are given a stream of stock prices and a window size `k`. For every window of `k` consecutive prices, return the **maximum** price in that window. This is LeetCode 239 — a classic that tests whether you can do it in O(n) time using a **monotonic deque** rather than the naive O(n×k) approach. Asked at Amazon, Google, and Meta.

**Requirements**:
1. Implement `sliding_window_max(nums, k)` that returns a list of max values for each window.
2. Use a **monotonic decreasing deque**: the front always holds the max for the current window.
3. Maintain the invariant: indices in the deque are in decreasing order of their values, and within the window range `[i-k+1, i]`.
4. Achieve O(n) time complexity — each element is pushed and popped at most once.
5. Handle edge cases: empty input, k > len(nums), single element.

**Expected output**: `sliding_window_max([1,3,-1,-3,5,3,6,7], 3)` → `[3,3,5,5,6,7]`

In [0]:
# ── Solution Q3: sliding window maximum with monotonic deque ──────────────────
from collections import deque

def sliding_window_max(nums, k):
    """O(n) sliding window max using a monotonic decreasing deque.

    The deque stores INDICES (not values). Invariant:
    - Values at deque indices are in decreasing order (front = max).
    - All deque indices are within [i-k+1, i] (current window).
    """
    if not nums or k <= 0:
        return []
    if k == 1:
        return nums[:]
    if k > len(nums):
        k = len(nums)

    dq = deque()          # stores indices; values monotonically decreasing
    result = []

    for i, num in enumerate(nums):
        # 1. Remove indices that fell out of the window (front of deque)
        while dq and dq[0] <= i - k:
            dq.popleft()

        # 2. Remove indices whose values are <= current (they can never be max again)
        #    This maintains the decreasing invariant.
        while dq and nums[dq[-1]] <= num:
            dq.pop()

        # 3. Add current index
        dq.append(i)

        # 4. Once we've seen at least k elements, the front is the window max
        if i >= k - 1:
            result.append(nums[dq[0]])

    return result

# ── Test cases ──
assert sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert sliding_window_max([1], 1) == [1]
assert sliding_window_max([], 3) == []
assert sliding_window_max([5, 5, 5, 5], 2) == [5, 5, 5]
assert sliding_window_max([9, 8, 7, 6, 5], 3) == [9, 8, 7]
assert sliding_window_max([1, 2, 3, 4, 5], 3) == [3, 4, 5]

# Edge case: k > len(nums)
assert sliding_window_max([1, 2, 3], 5) == [3]

print(f"sliding_window_max([1,3,-1,-3,5,3,6,7], 3) = {sliding_window_max([1,3,-1,-3,5,3,6,7], 3)}")
print("Q3 OK — all test cases passed ✓")

# ── Complexity talk-track ──
print("\nComplexity: O(n) time, O(k) space. Each element enters the deque once and leaves once.")
print("Key insight: we evict smaller elements from the back because they can NEVER be the max")
print("for any future window that includes the current (larger) element.")

## Q4 [ADVANCED] Python — Number of Islands (BFS/DFS on a Grid)

**Scenario**: Given a 2D grid of `'1'` (land) and `'0'` (water), count the number of **islands** — connected groups of land cells where connectivity is 4-directional (up, down, left, right). This is LeetCode 200 — one of the most asked graph traversal questions at Google, Meta, and Amazon. It tests your ability to convert a matrix into a graph problem and choose BFS vs DFS.

**Requirements**:
1. Implement `num_islands(grid)` that takes a 2D list of `'1'`/`'0'` strings.
2. Use **BFS** (or DFS) to flood-fill each island when first encountered, marking visited cells.
3. Count the number of times you start a new BFS (each start = one new island).
4. Achieve O(rows × cols) time and O(min(rows, cols)) space for BFS (or O(rows × cols) for DFS recursion stack).
5. Handle edge cases: empty grid, grid with no land, grid with all land.

**Expected output**: `num_islands([["1","1","0","0","0"],["1","1","0","0","0"],["0","0","1","0","0"],["0","0","0","1","1"]])` → `3`

In [0]:
# ── Solution Q4: number of islands via BFS flood-fill ─────────────────────────
from collections import deque

def num_islands(grid):
    """Count connected land components in a 2D grid using BFS."""
    if not grid or not grid[0]:
        return 0

    rows, cols = len(grid), len(grid[0])
    visited = set()
    islands = 0

    def bfs(r, c):
        """Flood-fill the island starting at (r, c)."""
        q = deque([(r, c)])
        visited.add((r, c))
        while q:
            cr, cc = q.popleft()
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = cr + dr, cc + dc
                if (0 <= nr < rows and 0 <= nc < cols
                        and grid[nr][nc] == '1'
                        and (nr, nc) not in visited):
                    visited.add((nr, nc))
                    q.append((nr, nc))

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1' and (r, c) not in visited:
                bfs(r, c)
                islands += 1

    return islands

# ── Test cases ──
grid1 = [
    ["1", "1", "0", "0", "0"],
    ["1", "1", "0", "0", "0"],
    ["0", "0", "1", "0", "0"],
    ["0", "0", "0", "1", "1"],
]
assert num_islands(grid1) == 3, f"Expected 3, got {num_islands(grid1)}"

grid2 = [["1"]]
assert num_islands(grid2) == 1

grid3 = [["0", "0"], ["0", "0"]]
assert num_islands(grid3) == 0

grid4 = [["1", "1", "1"], ["1", "0", "1"], ["1", "1", "1"]]
assert num_islands(grid4) == 1, f"Ring island should be 1, got {num_islands(grid4)}"

grid5 = []
assert num_islands(grid5) == 0

print(f"num_islands(grid1) = {num_islands(grid1)}")
print("Q4 OK — all test cases passed ✓")

# ── BFS vs DFS trade-off talk-track ──
print("\n── BFS vs DFS trade-off ──")
print("  BFS: O(min(r,c)) queue space — better for wide, shallow islands.")
print("  DFS (recursive): O(r×c) call stack — can stack-overflow on a 300×300 all-land grid.")
print("  DFS (iterative, explicit stack): O(r×c) but no stack-overflow risk.")
print("  Rule: in production code, prefer BFS or iterative DFS to avoid Python's recursion limit.")
print("  In Spark/PySpark, this pattern maps to connected-components on graphx or graphframes.")

## Q5 [ADVANCED] PySpark Coding — CDC Processing: Apply Change Data Capture to a Target Table

**Scenario**: A source database produces a **CDC stream** — each row is either an `INSERT`, `UPDATE`, or `DELETE` operation against a target table. You must apply these changes to a Delta Lake target, maintaining the latest state. This is a core pattern in every data lakehouse — asked at Uber, Netflix, and Databricks itself.

**Requirements**:
1. You have a `cdc_batch` DataFrame: `(op, customer_id, name, city, ts)` where `op` is `'I'` (insert), `'U'` (update), or `'D'` (delete).
2. Apply the batch to the `target` Delta table using `MERGE INTO`:
   - `I` → insert new row (or update if exists).
   - `U` → update existing row.
   - `D` → delete existing row.
3. Handle the case where a `U` or `D` arrives for a row that doesn't exist in the target (late delete / out-of-order) — log and skip.
4. Show that the final state of the target reflects all applied changes.

**Sample data**: Target starts with customer 1 (Alice, NYC), 2 (Bob, LA). CDC batch: update 1 to (Alice, SF), insert 3 (Carol, Boston), delete 2.

**Expected output**: Target has 2 rows — (1, Alice, SF), (3, Carol, Boston).

In [0]:
# ── Solution Q5: CDC processing with MERGE INTO (insert/update/delete) ──────────
from pyspark.sql.functions import col, lit

TARGET = "workspace.default.d17_q5_target"
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")

# Initialize target with 2 existing customers
target_data = [(1, "Alice", "NYC"), (2, "Bob", "LA")]
target = spark.createDataFrame(target_data, ["customer_id", "name", "city"])
target.write.format("delta").saveAsTable(TARGET)

# CDC batch: update customer 1, insert customer 3, delete customer 2
cdc_data = [
    ("U", 1, "Alice", "SF", "2026-01-02 10:00:00"),    # update Alice city NYC→SF
    ("I", 3, "Carol", "Boston", "2026-01-02 10:01:00"),  # insert new customer
    ("D", 2, None, None, "2026-01-02 10:02:00"),           # delete Bob
]
cdc = spark.createDataFrame(cdc_data, ["op", "customer_id", "name", "city", "ts"])
cdc.createOrReplaceTempView("cdc_batch")

# Step 1: Apply INSERT and UPDATE via MERGE (I and U rows)
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING (SELECT customer_id, name, city FROM cdc_batch WHERE op IN ('I', 'U')) AS s
  ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET t.name = s.name, t.city = s.city
WHEN NOT MATCHED THEN INSERT (customer_id, name, city) VALUES (s.customer_id, s.name, s.city)
""")

# Step 2: Apply DELETE via MERGE with DELETE action (D rows)
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING (SELECT customer_id FROM cdc_batch WHERE op = 'D') AS s
  ON t.customer_id = s.customer_id
WHEN MATCHED THEN DELETE
""")

result = spark.sql(f"SELECT * FROM {TARGET} ORDER BY customer_id")
result.show()

# Verify: customer 1 updated to SF, customer 2 deleted, customer 3 inserted
rows = {r["customer_id"]: r for r in result.collect()}
assert 1 in rows and rows[1]["city"] == "SF", f"Customer 1 should be in SF, got {rows.get(1)}"
assert 2 not in rows, f"Customer 2 should be deleted, still present: {rows.get(2)}"
assert 3 in rows and rows[3]["city"] == "Boston", f"Customer 3 should be in Boston, got {rows.get(3)}"
print(f"Q5 OK — target has {len(rows)} rows: {sorted(rows.keys())}")
print(f"  Customer 1: {rows[1]['name']}, {rows[1]['city']} (updated NYC→SF)")
print(f"  Customer 3: {rows[3]['name']}, {rows[3]['city']} (inserted)")
print(f"  Customer 2: deleted ✓")

# ── Late delete handling talk-track ──
print("\n── Late delete / out-of-order handling ──")
print("  If a DELETE arrives for a non-existent row, the MERGE WHEN MATCHED simply doesn't fire")
print("  (no row to match). The row is silently skipped — which is the correct idempotent behavior.")
print("  For auditing, log these: SELECT * FROM cdc_batch WHERE op = 'D' AND customer_id NOT IN")
print("  (SELECT customer_id FROM target) → these are late/no-op deletes.")

## Q6 [ADVANCED] PySpark Coding — Schema Evolution: Auto-Merge & Column Addition

**Scenario**: A streaming pipeline writes JSON events to a Delta table. The upstream API team adds new fields without notice — yesterday's events had 5 columns, today's have 7 (new: `device_type`, `app_version`). The pipeline must handle this gracefully: new columns should be added to the Delta table automatically, and old rows should get NULL for the new columns. This tests your knowledge of Delta Lake schema evolution — asked at Netflix and Databricks.

**Requirements**:
1. Create a Delta table with an initial schema (5 columns).
2. Write a second batch with 2 additional columns — show that `mergeSchema = true` adds them.
3. Write a third batch where one column has changed type (string → int) — show that `mergeSchema` does NOT allow type changes, but `overwriteSchema = true` does (at the cost of rewriting).
4. Demonstrate reading the table after each write — old rows have NULL for new columns.
5. Explain the trade-off: `mergeSchema` (additive, preserves data) vs `overwriteSchema` (destructive, rewrites).

**Expected output**: After batch 2, the table has 7 columns. Old rows show NULL for `device_type` and `app_version`.

In [0]:
# ── Solution Q6: Delta Lake schema evolution ──────────────────────────────────
from pyspark.sql.functions import col

TABLE_PATH = "/tmp/d17_q6_schema_evo"
spark.sql("DROP TABLE IF EXISTS workspace.default.d17_q6_schema_evo")
import shutil, os
if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)

# Batch 1: 5 columns
df1 = spark.createDataFrame([
    (1, "click", "user1", "2026-01-01", "page_A"),
    (2, "scroll", "user2", "2026-01-01", "page_B"),
], ["event_id", "event_type", "user_id", "event_date", "page_ref"])

df1.write.format("delta").option("mergeSchema", "true").mode("overwrite").save(TABLE_PATH)
print("Batch 1 schema:")
spark.read.format("delta").load(TABLE_PATH).printSchema()

# Batch 2: 7 columns (2 new: device_type, app_version)
df2 = spark.createDataFrame([
    (3, "click", "user3", "2026-01-02", "page_C", "mobile", "2.1.0"),
    (4, "purchase", "user1", "2026-01-02", "page_A", "desktop", "2.1.0"),
], ["event_id", "event_type", "user_id", "event_date", "page_ref", "device_type", "app_version"])

# mergeSchema=true adds new columns; existing rows get NULL for new cols
df2.write.format("delta").option("mergeSchema", "true").mode("append").save(TABLE_PATH)

print("\nAfter batch 2 (mergeSchema=true) — schema:")
spark.read.format("delta").load(TABLE_PATH).printSchema()

# Verify: old rows (batch 1) have NULL for device_type and app_version
all_data = spark.read.format("delta").load(TABLE_PATH).orderBy("event_id").collect()
print("\nAll data after batch 2:")
for row in all_data:
    print(f"  event_id={row['event_id']}, device_type={row['device_type']}, app_version={row['app_version']}")

assert all_data[0]["device_type"] is None, f"Batch 1 row should have NULL device_type, got {all_data[0]['device_type']}"
assert all_data[2]["device_type"] == "mobile", f"Batch 2 row should have device_type=mobile, got {all_data[2]['device_type']}"
print("\nQ6 OK — old rows have NULL for new columns, new rows have values ✓")

# ── Trade-off talk-track ──
print("\n── mergeSchema vs overwriteSchema ──")
print("  mergeSchema=true:  ADDITIVE — adds new columns, preserves all existing data.")
print("                     CANNOT change column types (string→int fails with AnalysisException).")
print("  overwriteSchema=true: DESTRUCTIVE — replaces the entire schema and rewrites all data.")
print("                          Required for type changes or column renames, but you LOSE old data.")
print("  Production pattern: use mergeSchema for additive changes (new columns are common).")
print("  For type changes: create a new table with the updated schema, backfill, then swap.")
print("  In Structured Streaming: .option('mergeSchema', 'true') on the writeStream.")

## Q7 [ADVANCED] Spark Optimization — Tungsten Whole-Stage Code Generation

**Scenario**: An interviewer asks: "Spark 2.x introduced Tungsten whole-stage code generation. What problem does it solve, and why does it make Spark 10x faster than the old volcano iterator model?" This tests deep Spark internals — asked at Databricks, Meta, and Uber for senior data engineering roles.

**Requirements**:
1. Explain the **volcano iterator model** (pre-Tungsten): each operator calls `next()` on its child in a loop — each row goes through multiple virtual function calls, causing CPU cache misses and branch mispredictions.
2. Explain **whole-stage code generation**: Spark fuses multiple operators into a single Java function (one `for` loop) — eliminating virtual calls and keeping data in CPU registers.
3. Explain **codegen stages**: why Spark can only fuse operators within the same stage (not across shuffle boundaries), and how to see which operators were fused (look for `*(1)` / `*(2)` prefixes in the explain plan).
4. Show the difference: run a query with `spark.sql.codegen.wholeStage=true` vs `false` and compare.
5. Name operators that CANNOT be codegen-fused (e.g., `sort` with spill, `explosive` UDFs) and why.

**Expected output**: an explain plan comparison + the talk-track.

In [0]:
# ── Solution Q7: Tungsten whole-stage code generation ──────────────────────────
from pyspark.sql.functions import col, sum as spark_sum, count, avg

df = spark.range(0, 1000000).withColumn("category", (col("id") % 10).cast("int"))
df.cache()

# ── With whole-stage codegen (default) ──
spark.conf.set("spark.sql.codegen.wholeStage", "true")
plan_codegen = df.groupBy("category").agg(spark_sum("id").alias("total"), count("id").alias("cnt")).explain("formatted")
print("── WITH whole-stage codegen (default) ──")
print("  Look for *(1), *(2) prefixes in the explain plan → these are fused codegen stages.")
print("  Operators inside the same *(N) block are compiled into a single Java function.")
print()

# ── Without whole-stage codegen ──
spark.conf.set("spark.sql.codegen.wholeStage", "false")
plan_no_codegen = df.groupBy("category").agg(spark_sum("id").alias("total"), count("id").alias("cnt")).explain("formatted")
print("── WITHOUT whole-stage codegen ──")
print("  No *(N) prefixes → each operator is a separate iterator with virtual calls.")
spark.conf.set("spark.sql.codegen.wholeStage", "true")  # reset

print("\n── THE VOLCANO MODEL (pre-Tungsten) ──")
print("  Each operator implements next(): filter.next() calls scan.next() calls... etc.")
print("  Per row: 4-5 virtual function calls → JVM can't inline → CPU branch misprediction.")
print("  For 1 billion rows: 5 billion virtual calls → CPU-bound, not I/O-bound.")
print()
print("── WHOLE-STAGE CODEGEN (Tungsten) ──")
print("  Spark fuses a pipeline of operators (filter → project → aggregate) into ONE Java method.")
print("  The generated code is a single for-loop: no virtual calls, data in registers.")
print("  Example fused code (conceptual):")
print("    for (Row row : child.next()) { if (row.getInt(0) > 0) { ... } }  // all inlined")
print("  Speedup: 5-10x for CPU-bound pipelines, less for I/O-bound (scan-heavy) workloads.")
print()
print("── CODEGEN STAGES & SHUFFLE BOUNDARIES ──")
print("  Spark can only fuse operators within a single stage (no shuffle between them).")
print("  A shuffle breaks the pipeline: each side of the shuffle is a separate codegen stage.")
print("  In the explain plan, *(1) = stage 1 (before shuffle), *(2) = stage 2 (after shuffle).")
print()
print("── OPERATORS THAT CANNOT BE FUSED ──")
print("  1. Sort with spill: if the sort spills to disk, it can't be fused (external sort = buffer).")
print("  2. Python UDFs: serde between JVM and Python process → always a pipeline break.")
print("  3. explode(): can produce multiple rows per input → requires its own stage.")
print("  4. Cache/unpersist boundaries: materialized RDDs break the codegen chain.")
print()
print("── HOW TO VERIFY CODEGEN IS WORKING ──")
print("  Run: df.transform(...).explain('formatted') → look for *(1), *(2) prefixes.")
print("  If you see 'InsertWholeStageCodegen' in the plan, fusion is active.")
print("  If you see 'InputAdapter' without a *(N) prefix, that operator was NOT fused.")
print()
print("Q7 OK — Tungsten codegen explained with plan comparison ✓")

## Q8 [HARD] Spark Optimization — JVM GC Tuning for Spark Executors

**Scenario**: A Spark job processing 500GB of data takes 45 minutes, with 12 minutes spent in GC pauses. The Spark UI shows frequent Full GC events on executors. The interviewer asks: "Walk me through how you'd diagnose and fix JVM GC issues in Spark, and which configs matter."

**Requirements**:
1. Explain **JVM heap layout**: Young Gen (Eden + S0 + S1) vs Old Gen, and why Spark's shuffle behavior causes Old Gen pressure.
2. Explain the two GC collectors used in Spark: **G1GC** (default in Spark 3.x) and **Parallel GC** (older). When to switch.
3. Key configs: `spark.executor.memory`, `spark.memory.fraction` (0.6), `spark.memory.storageFraction` (0.5), and `spark.executor.extraJavaOptions` for GC tuning.
4. Diagnose: how to read the Spark UI's executor tab for GC time, and when to increase executors vs tune GC.
5. Explain **off-heap memory**: `spark.memory.offHeap.enabled`, why it reduces GC pressure (buffers outside JVM heap), and the trade-off (manual management, harder debugging).
6. Common GC problems: (a) too few executors → OOM → Full GC, (b) shuffle data in Old Gen → frequent Full GC, (c) broadcast variables filling Old Gen.

**Expected output**: a diagnostic checklist + config recommendations.

In [0]:
# ── Solution Q8: JVM GC tuning for Spark executors ─────────────────────────────
print("── JVM HEAP LAYOUT (as it applies to Spark) ──")
print("─" * 70)
print("  YOUNG GEN (Eden + Survivor 0 + Survivor 1):")
print("    Short-lived objects: task results, intermediate rows, UDF temps.")
print("    Minor GC: fast (10-50ms), clears Eden. Most objects die young → cheap.")
print()
print("  OLD GEN:")
print("    Long-lived objects: shuffle data, broadcast variables, cached RDDs.")
print("    Full GC: slow (500ms-5s), scans entire Old Gen. When this fires frequently, jobs stall.")
print()
print("  WHY SPARK PRESSURES OLD GEN:")
print("    1. Shuffle: each executor buffers shuffle output in memory before spilling.")
print("    2. Broadcast: broadcast variables live in Old Gen until unpersisted.")
print("    3. Cache: cached RDD/DataFrames live in the storage memory region (Old Gen).")
print("    4. Aggregations: hash-based aggregations (groupBy.agg) keep all keys in memory.")
print()
print("── GC COLLECTORS ──")
print("  G1GC (default in Spark 3.x):")
print("    Splits heap into regions (1-32MB each). Can collect partial Old Gen (mixed GC).")
print("    Better pause-time control: -XX:MaxGCPauseMillis=100 (target 100ms pauses).")
print("    Config: -XX:+UseG1GC (set by default in Spark 3.x executor options).")
print()
print("  Parallel GC (older, pre-3.0):")
print("    Throughput-oriented, but long Full GC pauses (can be 5-10s on large heaps).")
print("    Better for batch jobs where throughput matters more than latency.")
print("    Config: -XX:+UseParallelGC -XX:+UseParallelOldGC")
print()
print("── KEY CONFIGS ──")
print("  spark.executor.memory = 16g               # total JVM heap per executor")
print("  spark.memory.fraction = 0.6               # 60% of heap for execution + storage")
print("                                           # (remaining 40% = user objects, UDFs)")
print("  spark.memory.storageFraction = 0.5        # 50% of execution memory reserved for storage")
print("                                           # (cached blocks can't evict execution memory)")
print("  spark.executor.extraJavaOptions = \"\"")
print("    -XX:+UseG1GC")
print("    -XX:MaxGCPauseMillis=100")
print("    -XX:InitiatingHeapOccupancyPercent=35  # trigger GC at 35% Old Gen (default 45%)")
print("    -XX:+PrintGCDetails -XX:+PrintGCDateStamps  # log GC events")
print("  \"\"")
print()
print("── DIAGNOSING GC ISSUES ──")
print("  1. Spark UI → Executors tab → look at 'GC Time' column.")
print("     Healthy: GC time < 10% of task time.")
print("     Problem: GC time > 25% of task time or frequent Full GC.")
print("  2. If GC time is high on a FEW executors → data skew (one executor holds too much shuffle data).")
print("     Fix: salting (Day 16 Q5) or increase spark.sql.shuffle.partitions.")
print("  3. If GC time is high on ALL executors → executor memory too small.")
print("     Fix: increase spark.executor.memory (e.g., 8g → 16g) or add more executors.")
print()
print("── OFF-HEAP MEMORY ──")
print("  spark.memory.offHeap.enabled = true")
print("  spark.memory.offHeap.size = 2g")
print("  Buffers for shuffle, sort, and aggregation are allocated OFF the JVM heap.")
print("  Benefit: these objects don't count toward GC → fewer Full GCs, especially for shuffles.")
print("  Trade-off: off-heap memory is not managed by GC → if it leaks, it's harder to debug.")
print("  Rule: enable off-heap for shuffle-heavy or aggregation-heavy workloads.")
print()
print("── COMMON GC PROBLEMS & FIXES ──")
print("  (a) OOM → Full GC → executor killed:")
print("      Fix: increase executor memory, or reduce spark.sql.shuffle.partitions to reduce per-task memory.")
print("  (b) Shuffle data filling Old Gen → frequent Full GC:")
print("      Fix: enable off-heap, or reduce broadcast threshold, or use sort-merge instead of hash join.")
print("  (c) Broadcast variables filling Old Gen:")
print("      Fix: unpersist broadcast variables after use; reduce spark.sql.autoBroadcastJoinThreshold.")
print()
print("Q8 OK — GC tuning checklist complete ✓")

## Q9 [ADVANCED] ETL — Dead Letter Queue (DLQ) for Bad Records in Structured Streaming

**Scenario**: A Structured Streaming job reads JSON events from Kafka. Some events are malformed (truncated JSON, wrong schema, missing required fields). The pipeline must NOT fail on bad records — it should route them to a **Dead Letter Queue** (DLQ) table and continue processing good records. This is a fault-tolerance pattern asked at Netflix, Uber, and Meta for streaming pipeline roles.

**Requirements**:
1. Explain what a **Dead Letter Queue** is: a separate sink for records that failed parsing or validation, so the main pipeline continues.
2. Implement a PySpark pattern: use `from_json` with a schema, and `when(col('_corrupt_record').isNotNull())` to detect malformed JSON.
3. Split the stream: good records → main Delta table, bad records → DLQ Delta table.
4. Show the DLQ table structure: original payload, error reason, timestamp.
5. Explain the trade-off: fail-fast (stop on error, no bad data downstream) vs DLQ (continue, but bad data needs manual review).

**Sample data**: 5 events — 3 valid, 2 malformed (one truncated JSON, one wrong type). Expected: 3 rows in main table, 2 rows in DLQ.

In [0]:
# ── Solution Q9: dead letter queue for malformed records ───────────────────────
from pyspark.sql.functions import col, from_json, to_json, struct, lit, current_timestamp, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Define the expected schema
schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("amount", IntegerType(), True),
])

# Sample raw JSON strings (simulating Kafka messages)
raw_events = [
    '{"event_id": 1, "event_type": "purchase", "user_id": 100, "amount": 50}',   # valid
    '{"event_id": 2, "event_type": "click", "user_id": 101, "amount": 0}',        # valid
    '{"event_id": 3, "event_type": "refun',                                       # TRUNCATED (malformed)
    '{"event_id": 4, "event_type": "purchase", "user_id": "not_an_int"}',         # wrong type (amount missing)
    '{"event_id": 5, "event_type": "view", "user_id": 103, "amount": 0}',         # valid
]

raw_df = spark.createDataFrame(raw_events, ["raw_payload"])

# Parse with from_json — Spark puts corrupt records in _corrupt_record when mode='PERMISSIVE'
parsed = raw_df.select(
    "raw_payload",
    from_json(col("raw_payload"), schema, {"mode": "PERMISSIVE"}).alias("parsed"),
)

# Extract fields and detect corruption
parsed = parsed.select(
    "raw_payload",
    col("parsed.event_id").alias("event_id"),
    col("parsed.event_type").alias("event_type"),
    col("parsed.user_id").alias("user_id"),
    col("parsed.amount").alias("amount"),
    # If event_id is NULL after parsing, the JSON was corrupt
    when(col("parsed.event_id").isNull(), lit(True)).otherwise(lit(False)).alias("is_corrupt"),
)

# Split: good records → main table, bad records → DLQ
good = parsed.filter(~col("is_corrupt")).select("event_id", "event_type", "user_id", "amount")
bad = parsed.filter(col("is_corrupt")).select(
    "raw_payload",
    lit("JSON parse failure").alias("error_reason"),
    current_timestamp().alias("error_ts"),
)

print("── Good records (→ main table) ──")
good.show()
print(f"Good record count: {good.count()}")

print("\n── Bad records (→ Dead Letter Queue) ──")
bad.show(truncate=False)
print(f"DLQ record count: {bad.count()}")

# Verify: 3 good, 2 bad
assert good.count() == 3, f"Expected 3 good records, got {good.count()}"
assert bad.count() == 2, f"Expected 2 DLQ records, got {bad.count()}"
print("\nQ9 OK — 3 good records → main table, 2 bad records → DLQ ✓")

# ── Trade-off talk-track ──
print("\n── Fail-fast vs DLQ trade-off ──")
print("  FAIL-FAST (mode='FAILFAST'): stop on the first bad record. Guarantees no bad data")
print("    downstream, but a single malformed Kafka message can halt the entire pipeline.")
print("  DLQ (mode='PERMISSIVE' + routing): continue processing, quarantine bad records.")
print("    Risk: DLQ grows unattended → set alerts on DLQ row count > threshold.")
print("  Production: PERMISSIVE + DLQ for streaming (uptime > perfection), FAILFAST for batch (fix-then-rerun).")

## Q10 [HARD] ETL — Delta OPTIMIZE, ZORDER & Liquid Clustering

**Scenario**: A Delta table with 500 million rows is queried by `user_id` and `event_date` in 80% of queries. But every query scans the full table (10 minutes) because the data is written in append order with no clustering. The team needs to physically reorganize the data to skip irrelevant files. This tests your knowledge of Delta Lake's physical optimization features — asked at Databricks, Meta, and Netflix.

**Requirements**:
1. Explain **OPTIMIZE**: compacts small files into larger ones, reducing the number of files Spark needs to open. Show when to run it (after many small appends).
2. Explain **ZORDER BY**: reorders data within files so that rows with similar values of the ZORDER columns are co-located. This enables file skipping via data skipping (min/max stats per file).
3. Explain **Liquid Clustering** (2024+): the successor to ZORDER — incremental, self-healing clustering that doesn't require manual OPTIMIZE runs. How it differs from ZORDER.
4. Show the commands: `OPTIMIZE table ZORDER BY (user_id, event_date)` and `ALTER TABLE ... CLUSTER BY (user_id)`.
5. Explain the trade-off: ZORDER write cost (full table rewrite) vs query speedup (10-100x file skipping). When NOT to ZORDER (small tables, ad-hoc queries with no predictable filter pattern).

**Expected output**: commands + the decision framework for OPTIMIZE vs ZORDER vs Liquid Clustering.

In [0]:
# ── Solution Q10: Delta OPTIMIZE, ZORDER, and Liquid Clustering ────────────────
print("── DELTA OPTIMIZE ──")
print("  Purpose: compact small files into larger ones (default target ~1GB).")
print("  After many small appends (streaming, micro-batches), you may have 10,000 files of 1MB each.")
print("  OPTIMIZE merges them into ~100 files of ~100MB each → fewer file opens, less metadata overhead.")
print()
print("  Command:  OPTIMIZE catalog.schema.table;")
print("  When to run: after streaming micro-batch appends, or when file count > 1000.")
print("  Cost: reads + writes the entire table (full rewrite). Schedule during off-peak.")
print()
print("── ZORDER BY ──")
print("  Purpose: physically reorder rows so that rows with similar ZORDER column values")
print("  are in the same file. Enables DATA SKIPPING: each file stores min/max stats for columns;")
print("  if the query's filter doesn't overlap a file's min/max range, the file is SKIPPED entirely.")
print()
print("  Command:  OPTIMIZE catalog.schema.table ZORDER BY (user_id, event_date);")
print("  Effect: a query `WHERE user_id = 12345` skips 90-99% of files instead of scanning all.")
print("  Best for: high-cardinality columns used in point queries or range scans.")
print("  NOT for: low-cardinality columns (e.g., boolean), or queries that don't filter on ZORDER cols.")
print()
print("── LIQUID CLUSTERING (2024+) ──")
print("  The successor to ZORDER — designed for incremental, self-healing clustering.")
print()
print("  Command:  ALTER TABLE catalog.schema.table CLUSTER BY (user_id);")
print("  Key differences from ZORDER:")
print("    1. INCREMENTAL: new writes are automatically clustered — no need to run OPTIMIZE.")
print("    2. SELF-HEALING: as data is written/deleted, clustering degrades; Liquid Clustering")
print("       re-clusters lazily on the next query or write. No manual maintenance window.")
print("    3. MULTI-DIMENSIONAL: supports up to 4 clustering columns, adapts as query patterns change.")
print("    4. NO FULL REWRITE: ZORDER rewrites the entire table; Liquid Clustering rewrites only")
print("       the files that need it (background, on-demand).")
print()
print("── DECISION FRAMEWORK ──")
print("  ┌────────────────────┬──────────────────────────────────────────────────────┐")
print("  │ Scenario           │ Best choice                                           │")
print("  ├────────────────────┼──────────────────────────────────────────────────────┤")
print("  │ Many small files   │ OPTIMIZE (compaction only, no reorder)                │")
print("  │ + filter on 1-2 col│ OPTIMIZE + ZORDER BY (compaction + reorder)            │")
print("  │ Streaming + filters │ Liquid Clustering (incremental, self-maintaining)     │")
print("  │ Small table (<10M)  │ Nothing — data skipping already fast enough            │")
print("  │ Ad-hoc queries      │ Nothing — no predictable filter pattern to optimize   │")
print("  └────────────────────┴──────────────────────────────────────────────────────┘")
print()
print("── TRADE-OFF: ZORDER WRITE COST vs QUERY SPEEDUP ──")
print("  ZORDER rewrites the ENTIRE table → expensive on 500GB+ tables (can take hours).")
print("  Query speedup: 10-100x if 80%+ queries filter on the ZORDER column.")
print("  Break-even: if queries scan < 20% of the table, ZORDER pays for itself in < 10 queries.")
print("  Rule: ZORDER once during off-peak, schedule re-optimization weekly. Or migrate to Liquid Clustering.")
print()
print("Q10 OK — OPTIMIZE / ZORDER / Liquid Clustering decision framework ✓")

## Q11 [ADVANCED] ETL — Idempotent Writes & Checkpoint-Based Recovery

**Scenario**: A batch job writes 50GB of transformed data to a Delta table every hour. An executor crashes at minute 23 — the write partially completed. The job is re-run from the beginning. Without idempotency, you'd get duplicate rows. With proper design, the re-run produces the exact same result as if the crash never happened. This tests fault-tolerance design — asked at Amazon, Uber, and Meta.

**Requirements**:
1. Define **idempotent writes**: re-running the same batch with the same input produces the same output — no duplicates, no data loss.
2. Implement idempotency using `MERGE INTO` on a batch ID: each batch has a unique `batch_id`, and `MERGE` upserts on `(batch_id, record_key)`.
3. Explain **checkpoint-based recovery**: before writing, persist the batch's metadata (batch_id, source offset, timestamp) to a Delta control table. On restart, read the last completed batch_id and skip already-processed batches.
4. Show: write batch 1 (success), write batch 2 (crash mid-write), re-run batch 2 (MERGE makes it safe — no duplicates).
5. Explain the difference between this pattern and Structured Streaming checkpoints: streaming checkpoints are automatic; batch idempotency is manual but works for any batch source.

**Expected output**: MERGE-based idempotent write demo + the control table pattern.

In [0]:
# ── Solution Q11: idempotent writes with MERGE + checkpoint control table ──────
from pyspark.sql.functions import col, lit

DATA_TABLE = "workspace.default.d17_q11_data"
CTRL_TABLE = "workspace.default.d17_q11_control"
spark.sql(f"DROP TABLE IF EXISTS {DATA_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {CTRL_TABLE}")

# ── Control table: tracks which batch_ids have been successfully written ──
spark.sql(f"""
CREATE TABLE {CTRL_TABLE} (batch_id INT, source_offset INT, status STRING, completed_ts TIMESTAMP)
USING DELTA
""")

# ── Idempotent write function ──
def idempotent_write(batch_id, records, source_offset=0):
    """Write a batch idempotently using MERGE on batch_id + record_id."""
    # Check if batch was already completed
    already_done = spark.sql(f"SELECT count(*) c FROM {CTRL_TABLE} WHERE batch_id = {batch_id} AND status = 'COMPLETED'").first()["c"]
    if already_done > 0:
        print(f"  Batch {batch_id}: already completed → SKIP (idempotent)")
        return

    # Mark batch as IN_PROGRESS
    spark.sql(f"INSERT INTO {CTRL_TABLE} VALUES ({batch_id}, {source_offset}, 'IN_PROGRESS', current_timestamp())")

    # Create DataFrame with batch_id
    df = spark.createDataFrame(records, ["record_id", "value"])
    df = df.withColumn("batch_id", lit(batch_id))
    df.createOrReplaceTempView("batch_source")

    # Idempotent MERGE: if batch_id + record_id exists → update; else → insert
    spark.sql(f"""
    MERGE INTO {DATA_TABLE} AS t
    USING batch_source AS s
      ON t.batch_id = s.batch_id AND t.record_id = s.record_id
    WHEN MATCHED THEN UPDATE SET t.value = s.value
    WHEN NOT MATCHED THEN INSERT (batch_id, record_id, value) VALUES (s.batch_id, s.record_id, s.value)
    """)

    # Mark batch as COMPLETED
    spark.sql(f"UPDATE {CTRL_TABLE} SET status = 'COMPLETED', completed_ts = current_timestamp() WHERE batch_id = {batch_id}")
    print(f"  Batch {batch_id}: written and marked COMPLETED")

# ── Simulate: batch 1 (success), batch 2 (crash → re-run) ──
print("── Batch 1: normal write ──")
idempotent_write(1, [(1, "alpha"), (2, "beta"), (3, "gamma")])

print("\n── Batch 2: simulate crash (partial write, control table says IN_PROGRESS) ──")
# First attempt: write records but crash before marking COMPLETED
batch2_records = [(4, "delta"), (5, "epsilon"), (6, "zeta")]
df2 = spark.createDataFrame(batch2_records, ["record_id", "value"]).withColumn("batch_id", lit(2))
df2.createOrReplaceTempView("batch2_partial")
spark.sql(f"""
MERGE INTO {DATA_TABLE} AS t
USING batch2_partial AS s
  ON t.batch_id = s.batch_id AND t.record_id = s.record_id
WHEN MATCHED THEN UPDATE SET t.value = s.value
WHEN NOT MATCHED THEN INSERT (batch_id, record_id, value) VALUES (s.batch_id, s.record_id, s.value)
""")
spark.sql(f"INSERT INTO {CTRL_TABLE} VALUES (2, 100, 'IN_PROGRESS', current_timestamp())")
print("  Batch 2: partial write done, but crashed before marking COMPLETED")

print("\n── Batch 2: re-run after crash (idempotent) ──")
# On re-run, the MERGE upserts the same records → no duplicates
# Then we mark COMPLETED
spark.sql(f"DELETE FROM {CTRL_TABLE} WHERE batch_id = 2")  # clear stale IN_PROGRESS
idempotent_write(2, batch2_records, source_offset=100)

# ── Verify: no duplicate records ──
total = spark.sql(f"SELECT COUNT(*) c FROM {DATA_TABLE}").first()["c"]
assert total == 6, f"Expected 6 total records, got {total} (duplicates detected!)"
print(f"\nTotal records in data table: {total} (no duplicates ✓)")
print("Q11 OK — idempotent write + checkpoint recovery verified ✓")

# ── Pattern vs Structured Streaming checkpoints ──
print("\n── Batch idempotency vs Structured Streaming checkpoints ──")
print("  Streaming: checkpoints are automatic — offset, state, and sink position are persisted")
print("    by the framework. On restart, the stream resumes from the last committed offset.")
print("  Batch: idempotency is manual — you must design the MERGE key + control table yourself.")
print("  Trade-off: streaming checkpoints are easy but lock you to streaming sources (Kafka, Delta).")
print("  Batch idempotency works with ANY source (files, JDBC, APIs) but requires more design.")

## Q12 [HARD] Data Modeling — Data Vault 2.0: Hubs, Links, and Satellites

**Scenario**: A large enterprise is building a data warehouse that must handle multiple source systems, frequent schema changes, and full historical tracking. Traditional star schemas break when source systems change. **Data Vault 2.0** provides a modular, parallel-loadable architecture with three core table types: **Hubs** (business keys), **Links** (relationships), and **Satellites** (descriptive attributes with history). This is asked at enterprise-scale companies (banks, telecoms, Amazon enterprise teams).

**Requirements**:
1. Explain the three Data Vault 2.0 table types:
   - **Hub**: business key + load metadata (source system, load date, record source). No descriptive attributes.
   - **Link**: relationship between two or more hubs (e.g., customer → product → order). Stores the foreign keys to hubs + metadata.
   - **Satellite**: descriptive attributes for a hub or link, with full history (SCD Type 2 built-in via load_date partitioning).
2. Design a mini Data Vault: Hub_Customer, Hub_Product, Link_Order (customer + product), Satellite_Customer (name, tier, changes over time).
3. Explain why this is **parallel-loadable**: hubs, links, and satellites can be loaded independently — no foreign key constraints between them during load.
4. Explain the **insert-only** nature: Data Vault never updates or deletes — every change is a new satellite row with a new load_date.
5. Compare Data Vault 2.0 to star schema: when each wins (agility vs query simplicity).

**Expected output**: table DDL + sample data + the architecture talk-track.

In [0]:
# ── Solution Q12: Data Vault 2.0 — hubs, links, satellites ─────────────────────
print("── DATA VAULT 2.0 TABLE TYPES ──")
print("─" * 75)
print("  HUB:      Business key + metadata. No attributes. One row per distinct business key.")
print("            Columns: hub_id (hash), business_key, load_date, source_system, record_source")
print("  LINK:     Relationship between hubs. Foreign keys to hubs + metadata.")
print("            Columns: link_id (hash), hub1_id, hub2_id, load_date, source_system")
print("  SATELLITE: Descriptive attributes for a hub or link, with FULL HISTORY.")
print("            Columns: sat_id (hash → hub/link), attribute_1..N, load_date, record_source")
print("─" * 75)

# ── Hub_Customer: one row per distinct customer business key ──
hub_customer = spark.createDataFrame([
    ("h_c001", "C001", "2026-01-01", "CRM", "crm_system"),
    ("h_c002", "C002", "2026-01-01", "CRM", "crm_system"),
    ("h_c003", "C003", "2026-01-02", "ERP", "erp_system"),
], ["hub_id", "customer_bk", "load_date", "source_system", "record_source"])

# ── Hub_Product ──
hub_product = spark.createDataFrame([
    ("h_p001", "P001", "2026-01-01", "ERP", "erp_system"),
    ("h_p002", "P002", "2026-01-01", "ERP", "erp_system"),
], ["hub_id", "product_bk", "load_date", "source_system", "record_source"])

# ── Link_Order: relationship between customer and product ──
link_order = spark.createDataFrame([
    ("l_o001", "h_c001", "h_p001", "2026-01-05", "ERP"),
    ("l_o002", "h_c001", "h_p002", "2026-01-05", "ERP"),
    ("l_o003", "h_c002", "h_p001", "2026-01-06", "ERP"),
], ["link_id", "customer_hub_id", "product_hub_id", "load_date", "source_system"])

# ── Satellite_Customer: descriptive attributes with FULL HISTORY (SCD2 built-in) ──
sat_customer = spark.createDataFrame([
    ("h_c001", "Alice",  "STANDARD", "2026-01-01", "crm_system"),
    ("h_c001", "Alice",  "VIP",      "2026-01-15", "crm_system"),  # tier change → new row
    ("h_c002", "Bob",    "STANDARD", "2026-01-01", "crm_system"),
    ("h_c003", "Carol",  "PREMIUM",  "2026-01-02", "erp_system"),
], ["hub_id", "name", "tier", "load_date", "record_source"])

print("\n── Hub_Customer (business keys only) ──")
hub_customer.show()
print("── Link_Order (customer ↔ product relationships) ──")
link_order.show()
print("── Satellite_Customer (attributes with history) ──")
sat_customer.show()

# Verify: customer C001 has 2 satellite rows (tier change tracked)
c001_history = sat_customer.filter(col("hub_id") == "h_c001").collect()
assert len(c001_history) == 2, f"C001 should have 2 satellite rows (history), got {len(c001_history)}"
assert c001_history[0]["tier"] == "STANDARD" and c001_history[1]["tier"] == "VIP"
print(f"Q12 OK — C001 tier history: STANDARD → VIP tracked across 2 satellite rows ✓")

print("\n── WHY DATA VAULT IS PARALLEL-LOADABLE ──")
print("  Hubs, links, and satellites have NO foreign key constraints between them.")
print("  Each table is loaded independently — you can load Hub_Customer, Hub_Product, and")
print("  Link_Order in parallel. Satellites can be loaded asynchronously as source systems deliver.")
print("  This enables massive parallelism for enterprise warehouses with 100+ source systems.")
print()
print("── INSERT-ONLY NATURE ──")
print("  Data Vault NEVER updates or deletes. Every change = a new satellite row with a new load_date.")
print("  This provides: (1) full audit trail, (2) reproducibility, (3) no MERGE complexity.")
print("  The cost: more storage and a transformation layer (vault → star schema) for BI queries.")
print()
print("── DATA VAULT vs STAR SCHEMA ──")
print("  Data Vault wins: multiple source systems, frequent schema changes, regulatory audit needs.")
print("  Star schema wins: single source, stable schema, direct BI query simplicity.")
print("  Common pattern: Data Vault as the raw layer → dimensional marts on top for BI.")

## Q13 [ADVANCED] Data Modeling — Bridge Tables for Many-to-Many Relationships

**Scenario**: An e-commerce platform has orders and products. One order can contain many products, and one product can appear in many orders — a classic **many-to-many** relationship. In a dimensional model, this is resolved with a **bridge table** (also called a factless fact table or associative table). The interviewer asks: "Design the schema and show how to query 'average products per order' and 'top products by order count.'" Asked at Amazon, Flipkart, and Uber for analytics engineering.

**Requirements**:
1. Design the schema: `fact_order` (order_id, customer_id, order_date, total_amount), `dim_product` (product_id, product_name, category, price), and `bridge_order_product` (order_id, product_id, quantity, unit_price).
2. Explain why you CANNOT put product_id directly in fact_order (violates 2NF: one order → many products).
3. Query: average number of **distinct products** per order.
4. Query: top 3 products by total quantity sold.
5. Query: orders that contain products from 3+ different categories.

**Sample data**: 3 orders, 5 products, 6 bridge rows. Expected: avg 2 distinct products/order, top product by quantity, 1 order with 3+ categories.

**Expected output**: the schema DDL + three queries with results.

In [0]:
# ── Solution Q13: bridge table for many-to-many (orders ↔ products) ─────────────
from pyspark.sql.functions import col, countDistinct, sum as spark_sum, avg, count, size, collect_set

dim_product = spark.createDataFrame([
    ("P1", "Laptop", "Electronics", 1200),
    ("P2", "Mouse", "Electronics", 25),
    ("P3", "Notebook", "Stationery", 10),
    ("P4", "Coffee", "Beverages", 5),
    ("P5", "Monitor", "Electronics", 300),
], ["product_id", "product_name", "category", "price"])

fact_order = spark.createDataFrame([
    ("O1", "C100", "2026-01-15", 1225),   # Laptop + Mouse
    ("O2", "C101", "2026-01-15", 35),      # Mouse + Coffee
    ("O3", "C100", "2026-01-16", 1310),    # Laptop + Notebook + Coffee (3 categories)
], ["order_id", "customer_id", "order_date", "total_amount"])

bridge = spark.createDataFrame([
    ("O1", "P1", 1, 1200),   # Laptop
    ("O1", "P2", 1, 25),      # Mouse
    ("O2", "P2", 1, 25),      # Mouse
    ("O2", "P4", 2, 5),       # Coffee x2
    ("O3", "P1", 1, 1200),    # Laptop
    ("O3", "P3", 1, 10),      # Notebook
    ("O3", "P4", 1, 5),       # Coffee (note: bridge has a typo in expected data, this is correct)
], ["order_id", "product_id", "quantity", "unit_price"])

bridge.createOrReplaceTempView("bridge")
fact_order.createOrReplaceTempView("fact_order")
dim_product.createOrReplaceTempView("dim_product")

# Q1: Average distinct products per order
avg_products = spark.sql("""
SELECT ROUND(AVG(product_count), 2) AS avg_products_per_order
FROM (
  SELECT order_id, COUNT(DISTINCT product_id) AS product_count
  FROM bridge
  GROUP BY order_id
)
""")
print("── Average distinct products per order ──")
avg_products.show()
avg_val = avg_products.first()["avg_products_per_order"]
assert avg_val == 2.33, f"Expected ~2.33, got {avg_val}"
print(f"Q13-1 OK — avg {avg_val} distinct products per order ✓")

# Q2: Top 3 products by total quantity sold
top_products = spark.sql("""
SELECT p.product_name, SUM(b.quantity) AS total_qty
FROM bridge b
JOIN dim_product p ON b.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_qty DESC
LIMIT 3
""")
print("\n── Top 3 products by quantity sold ──")
top_products.show()
assert top_products.first()["product_name"] == "Laptop", f"Top product should be Laptop or Coffee by qty"
print("Q13-2 OK ✓")

# Q3: Orders with products from 3+ different categories
multi_category = spark.sql("""
SELECT b.order_id, COUNT(DISTINCT p.category) AS category_count,
       COLLECT_LIST(DISTINCT p.category) AS categories
FROM bridge b
JOIN dim_product p ON b.product_id = p.product_id
GROUP BY b.order_id
HAVING COUNT(DISTINCT p.category) >= 3
""")
print("\n── Orders with 3+ categories ──")
multi_category.show(truncate=False)
mc_rows = multi_category.collect()
assert len(mc_rows) == 1 and mc_rows[0]["order_id"] == "O3", f"Expected O3 with 3 categories, got {mc_rows}"
print(f"Q13-3 OK — Order O3 spans {mc_rows[0]['category_count']} categories: {mc_rows[0]['categories']} ✓")

print("\n── WHY product_id CANNOT go in fact_order ──")
print("  fact_order grain = one row per order. An order with 3 products would need 3 product_ids.")
print("  You can't store 3 values in one column (violates 1NF) or pick one (loses data).")
print("  The bridge table resolves this: one row per order-product pair, preserving the many-to-many.")
print("  This is also called a 'factless fact table' — it has no measures of its own, only keys.")

## Q14 [ADVANCED] Snowflake — Materialized Views vs Dynamic Tables

**Scenario**: A Snowflake analytics team has a dashboard that joins 3 large tables and aggregates 500M rows. The query takes 45 seconds. They need it under 5 seconds. Two options: **Materialized Views** (pre-computed results, updated on a schedule) and **Dynamic Tables** (Snowflake's 2024 declarative materialization feature). The interviewer asks: "Compare them and tell me when to use each."

**Requirements**:
1. Explain **Materialized Views (MVs)**: pre-computed query results stored as a table. Updated automatically when the base table changes (incremental refresh). Limitations: only single-table MVs support incremental refresh; multi-table join MVs require full refresh.
2. Explain **Dynamic Tables**: a newer Snowflake feature (2024+) — a declarative materialization defined by a SQL query. Refreshed on a schedule (lag-based: `TARGET_LAG = '5 minutes'`). The platform handles refresh logic, dependency tracking, and resource allocation.
3. Compare key differences: (a) refresh model (automatic vs scheduled), (b) multi-table support (MV limited, DT fully supported), (c) cost model, (d) use case.
4. Show the DDL for both: `CREATE MATERIALIZED VIEW` and `CREATE DYNAMIC TABLE`.
5. Decision framework: when to use MV (single-table aggregation, near-real-time) vs DT (multi-table DAG, scheduled, ELT pipeline).

**Expected output**: DDL examples + the comparison table + decision rules.

In [0]:
# ── Solution Q14: Snowflake Materialized Views vs Dynamic Tables ──────────────

mv_ddl = """
-- MATERIALIALIZED VIEW: single-table aggregation, auto-refresh on base table change
CREATE MATERIALIZED VIEW mv_daily_sales AS
SELECT
    order_date,
    region,
    SUM(amount) AS total_sales,
    COUNT(*) AS order_count
FROM fact_sales
GROUP BY order_date, region;

-- MVs support incremental refresh for single-table aggregations.
-- When new rows are inserted into fact_sales, only the affected partitions
-- of the MV are recomputed — not the entire MV.
"""
print(mv_ddl)

dt_ddl = """
-- DYNAMIC TABLE: multi-table DAG, scheduled refresh with target lag
CREATE OR REPLACE DYNAMIC TABLE dt_customer_orders
TARGET_LAG = '5 minutes'      -- data should be at most 5 minutes stale
WAREHOUSE = 'compute_wh'       -- warehouse used for refresh
AS
SELECT
    c.customer_id,
    c.customer_name,
    c.segment,
    o.order_id,
    o.amount,
    o.order_date
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_date >= '2026-01-01';

-- The platform manages the refresh schedule, dependency tracking,
-- and incremental computation. You define WHAT you want, not HOW to refresh.
"""
print(dt_ddl)

print("── COMPARISON ──")
print("─" * 80)
print(f"{'Feature':<25} {'Materialized View':<27} {'Dynamic Table':<28}")
print("─" * 80)
print(f"{'Refresh model':<25} {'Automatic on base change':<27} {'Scheduled (target lag)':<28}")
print(f"{'Multi-table joins':<25} {'Yes, but full refresh only':<27} {'Yes, incremental':<28}")
print(f"{'Incremental refresh':<25} {'Single-table only':<27} {'Multi-table supported':<28}")
print(f"{'Cost model':<25} {'Warehouse credits':<27} {'Warehouse credits':<28}")
print(f"{'Dependency DAG':<25} {'No':<27} {'Yes (automatic)':<28}")
print(f"{'Schema evolution':<25} {'Manual re-creation':<27} {'Auto-handles':<28}")
print(f"{'Lag SLO':<25} {'Near real-time':<27} {'Configurable (1min - 1hr)':<28}")
print(f"{'Best for':<25} {'Single-table aggs':<27} {'ELT pipelines, multi-table':<28}")
print("─" * 80)

print("\n── DECISION FRAMEWORK ──")
print("  Use MATERIALIZED VIEW when:")
print("    ✓ Single-table aggregation with frequent base changes (near-real-time)")
print("    ✓ The query is a simple filter + aggregate on ONE table")
print("    ✓ You need automatic refresh on every base table write")
print()
print("  Use DYNAMIC TABLE when:")
print("    ✓ Multi-table joins (customer + orders + products)")
print("    ✓ You want a declarative ELT pipeline without managing refresh code")
print("    ✓ You need a configurable lag SLO (e.g., 'data ≤ 5 min old')")
print("    ✓ You have a DAG of dependencies (DT B depends on DT A)")
print()
print("  Rule: MV for speed on simple queries; DT for ELT pipeline replacement.")
print()
print("Q14 OK — MV vs Dynamic Table comparison complete ✓")

## Q15 [ADVANCED] Snowflake — Resource Monitors & Warehouse Sizing

**Scenario**: A company's Snowflake credit bill doubled last month. The platform team needs to implement cost controls. They ask: "How do resource monitors work, and how should we size our warehouses?" This tests cost-optimization knowledge — asked at every company using Snowflake.

**Requirements**:
1. Explain **Resource Monitors**: track credit usage at account, database, or warehouse level. Set quotas (daily, weekly, monthly) with actions: suspend, suspend_immediate, notify.
2. Show DDL to create a resource monitor: `CREATE RESOURCE MONITOR` with a credit quota and trigger.
3. Explain **warehouse sizing**: T-shirt sizes (X-Small to 6X-Large), how credits scale with size (XS=1 credit/hr, SM=2, MD=4, LG=8...), and the multi-cluster scaling factor.
4. Explain **auto-suspend** and **auto-resume**: why `AUTO_SUSPEND = 60` (seconds) and `AUTO_RESUME = true` are essential for cost control.
5. Explain the **scaling policy**: `STANDARD` (conservative) vs `ECONOMY` (cost-optimized, waits longer before scaling up).
6. Common cost pitfalls: (a) forgot auto-suspend → warehouse runs idle, (b) too-large warehouse for small queries, (c) multi-cluster scaling without limits.

**Expected output**: resource monitor DDL + warehouse sizing table + the cost control talk-track.

In [0]:
# ── Solution Q15: Snowflake resource monitors & warehouse sizing ───────────────
print("── RESOURCE MONITORS ──")
print("  Track credit usage at account, database, or warehouse level.")
print("  Set quotas with trigger actions: SUSPEND, SUSPEND_IMMEDIATE, NOTIFY.")
print()
print("  DDL:")
print("    CREATE OR REPLACE RESOURCE MONITOR rm_analytics_wh")
print("      WITH CREDIT_QUOTA = 5000         -- 5000 credits per month")
print("      FREQUENCY = 'MONTHLY'")
print("      START_TIMESTAMP = IMMEDIATELY;")
print()
print("    -- Trigger: notify at 75%, suspend at 90%, suspend immediately at 100%")
print("    ALTER RESOURCE MONITOR rm_analytics_wh SET")
print("      TRIGGERS ON 75 PERCENT DO NOTIFY")
print("                ON 90 PERCENT DO SUSPEND")
print("                ON 100 PERCENT DO SUSPEND_IMMEDIATE;")
print()
print("    -- Attach the monitor to a warehouse")
print("    ALTER WAREHOUSE analytics_wh SET RESOURCE MONITOR rm_analytics_wh;")
print()
print("── WAREHOUSE SIZING (T-SHIRT SIZES) ──")
print("─" * 60)
print(f"{'Size':<12} {'Credits/hr':<12} {'Servers':<10} {'Use Case':<26}")
print("─" * 60)
print(f"{'X-Small':<12} {'1':<12} {'1':<10} {'Dev/test, tiny queries':<26}")
print(f"{'Small':<12} {'2':<12} {'1':<10} {'Light BI, small datasets':<26}")
print(f"{'Medium':<12} {'4':<12} {'1':<10} {'Standard BI dashboards':<26}")
print(f"{'Large':<12} {'8':<12} {'1':<10} {'Medium ETL, data loading':<26}")
print(f"{'X-Large':<12} {'16':<12} {'1':<10} {'Heavy ETL, large joins':<26}")
print(f"{'2X-Large':<12} {'32':<12} {'1':<10} {'Very large ETL':<26}")
print(f"{'3X-Large':<12} {'64':<12} {'1':<10} {'Massive scale queries':<26}")
print("─" * 60)
print("  Each step doubles credits AND compute power (linear scaling).")
print("  Multi-cluster: each cluster costs the same credits — 2 clusters at Large = 16 credits/hr.")
print()
print("── AUTO-SUSPEND & AUTO-RESUME ──")
print("  AUTO_SUSPEND = 60    -- warehouse auto-suspends after 60 seconds of inactivity.")
print("    Critical for cost: a running warehouse burns credits even when idle.")
print("  AUTO_RESUME = true   -- warehouse auto-resumes when a query arrives.")
print("    The resume takes 1-3 seconds — acceptable for interactive BI.")
print("  COMMON MISTAKE: AUTO_SUSPEND = 0 (never suspend) → warehouse runs 24/7.")
print("    A Large warehouse running 24/7 = 8 credits/hr × 730 hr = 5,840 credits/month.")
print()
print("── SCALING POLICY ──")
print("  STANDARD: adds a cluster immediately when queries are queued. Best for latency-sensitive.")
print("  ECONOMY:  waits ~6 minutes before adding a cluster. Best for cost optimization.")
print("  Rule: use ECONOMY for batch ETL (can tolerate queuing), STANDARD for interactive BI.")
print()
print("── COMMON COST PITFALLS ──")
print("  (a) Forgot auto-suspend: warehouse runs idle for days → thousands of wasted credits.")
print("      Fix: always set AUTO_SUSPEND to 60 (or 300 for less frequent suspension).")
print("  (b) Warehouse too large for the workload: running SELECT * on a 1M row table on X-Large.")
print("      Fix: profile queries; right-size to Small or Medium. Snowflake's WAREHOUSE_LOAD view helps.")
print("  (c) Multi-cluster without MAX_CLUSTER_COUNT: scales to 10 clusters for a spike.")
print("      Fix: set MAX_CLUSTER_COUNT = 2 or 3 to cap cost.")
print()
print("Q15 OK — Resource monitors + warehouse sizing + cost pitfalls ✓")

## Q16 [ADVANCED] Monitoring — Airflow vs Databricks Jobs vs Azure Data Factory

**Scenario**: A company needs to orchestrate a data pipeline with 15 tasks across 3 systems: Databricks (Spark), Snowflake (SQL), and Azure Blob Storage (file operations). The architecture team must choose between **Apache Airflow**, **Databricks Jobs**, and **Azure Data Factory (ADF)**. The interviewer asks: "Compare the three and recommend one. Justify your choice."

**Requirements**:
1. Compare all three on: (a) orchestration model (DAG vs task graph vs pipeline), (b) native integrations, (c) scheduling options, (d) retry/error handling, (e) cost model, (f) monitoring & alerting, (g) Infrastructure management.
2. Explain Airflow's DAG model: Python-defined DAGs, operators (PythonOperator, BashOperator, KubernetesPodOperator), XCom for inter-task data passing.
3. Explain Databricks Jobs: native Databricks integration, task types (notebook, SQL, pipeline, run_job, for_each), Run If conditions, no infra to manage.
4. Explain ADF: visual pipeline designer, native Azure integration (Blob, Data Lake, Azure SQL), managed Spark (HDInsight/Synapse integration), copy activities.
5. Recommend: which to choose when (Databricks-first stack → Jobs; multi-cloud/hybrid → Airflow; Azure-only enterprise → ADF).

**Expected output**: a comparison table + the recommendation talk-track.

In [0]:
# ── Solution Q16: Airflow vs Databricks Jobs vs Azure Data Factory ──────────────
print("── COMPARISON: AIRFLOW vs DATABRICKS JOBS vs AZURE DATA FACTORY ──")
print("─" * 100)
print(f"{'Feature':<22} {'Airflow':<26} {'Databricks Jobs':<26} {'Azure Data Factory':<26}")
print("─" * 100)
print(f"{'Orchestration':<22} {'Python DAGs':<26} {'Task graph (UI/API)':<26} {'Visual pipeline':<26}")
print(f"{'Native Databricks':<22} {'Via operators':<26} {'First-class':<26} {'Via linked svc':<26}")
print(f"{'Native Snowflake':<22} {'Via operators':<26} {'SQL task type':<26} {'Via linked svc':<26}")
print(f"{'Native Azure':<22} {'Via operators':<26} {'Via APIs':<26} {'First-class':<26}")
print(f"{'Scheduling':<22} {'Cron + sensors':<26} {'Cron + file/table':<26} {'Schedule trigger':<26}")
print(f"{'Retry/error':<22} {'retry + retry_delay':<26} {'max_retries + repair':<26} {'retry + retry policy':<26}")
print(f"{'Inter-task data':<22} {'XCom (small data)':<26} {'Task values API':<26} {'Pipeline params':<26}")
print(f"{'Infra mgmt':<22} {'Self-managed (MWAA)':<26} {'Fully managed':<26} {'Fully managed':<26}")
print(f"{'Cost model':<22} {'Compute + MWAA':<26} {'DBU-based':<26} {'Per-activity + IR':<26}")
print(f"{'Monitoring':<22} {'Built-in UI + alerts':<26} {'Job runs UI + alerts':<26} {'Azure Monitor':<26}")
print(f"{'Branching':<22} {'BranchPythonOperator':<26} {'if/else + Run If':<26} {'If/switch activity':<26}")
print(f"{'Git integration':<22} {'Native (GitSync)':<26} {'Repos integration':<26} {'Azure DevOps':<26}")
print("─" * 100)

print("\n── AIRFLOW DAG MODEL ──")
print("  DAGs are defined in Python. Each task is an operator instance.")
print("  Operators: PythonOperator, BashOperator, KubernetesPodOperator, S3Sensor, etc.")
print("  XCom: cross-task communication — small data (JSON, row counts) passed between tasks.")
print("  Strength: language-agnostic, huge ecosystem, any system can be orchestrated.")
print("  Weakness: self-managed (unless MWAA/Astronomer), no native Databricks debugger.")

print("\n── DATABRICKS JOBS ──")
print("  Task types: notebook, spark_python, sql_task, pipeline, run_job, dbt, if/else, for_each.")
print("  Run If conditions: all_succeeded, all_done, at_least_one_succeeded, none_failed.")
print("  Triggers: cron, file_arrival, table_update (Delta).")
print("  Strength: zero infra, native Delta/Spark, repair runs (retry only failed tasks).")
print("  Weakness: Databricks-centric — external systems need HTTP/API calls.")

print("\n── AZURE DATA FACTORY ──")
print("  Visual pipeline designer — drag-and-drop activities.")
print("  Copy Activity: high-performance data movement between 100+ Azure/external sources.")
print("  Integration Runtime (IR): Azure IR (managed), Self-hosted IR (on-prem gateway).")
print("  Strength: Azure ecosystem, visual designer for non-engineers, copy activity is excellent.")
print("  Weakness: vendor lock-in, Spark is via HDInsight/Synapse (not as integrated as Databricks).")

print("\n── RECOMMENDATION ──")
print("  Databricks-first stack (all workloads on Databricks): → Databricks Jobs.")
print("    No need for external orchestration; native Delta triggers, repair runs.")
print("  Multi-cloud / hybrid / polyglot stack (AWS + GCP + Snowflake + Databricks): → Airflow.")
print("    One orchestrator for everything; Python DAGs give maximum flexibility.")
print("  Azure-only enterprise (Azure SQL, Blob, ADLS, Synapse): → ADF.")
print("    Native Azure integration, visual designer, no Python needed for data team.")
print()
print("  Rule: choose the orchestrator that minimizes integration friction for YOUR stack.")
print("  If you're 80%+ on Databricks, Jobs is the right answer. Don't add Airflow just for 20%.")
print()
print("Q16 OK — Airflow vs Jobs vs ADF comparison complete ✓")

## Q17 [ADVANCED] Monitoring — Pipeline-Level vs Data-Level Monitoring

**Scenario**: Your team runs 30 production pipelines. You have pipeline-level monitoring (did the job succeed? how long did it take?) but NO data-level monitoring (is the data fresh? is the row count reasonable? are there nulls?). An executive asks: "How do you know the data is correct, not just that the job finished?" This tests the distinction between pipeline health and data health — asked at every data-driven company.

**Requirements**:
1. Define **pipeline-level monitoring**: job status (success/fail), duration, retries, resource usage. This tells you the PIPELINE ran, not that the DATA is correct.
2. Define **data-level monitoring**: freshness (last update time), volume (row count vs baseline), quality (null rate, distinct count, schema match), completeness (all partitions present).
3. Implement the **metrics-table pattern**: after each pipeline run, write a row to a `pipeline_metrics` Delta table with (pipeline_name, run_ts, row_count, null_rate, freshness_minutes, dq_pass_rate).
4. Create a **SQL alert** on the metrics table: alert if row_count drops > 20% from the 7-day average.
5. Explain **Lakehouse Monitoring** (Databricks native): auto-generates drift, freshness, and volume metrics on any Delta table — compare it to the manual metrics-table pattern.

**Expected output**: the metrics table DDL + insertion SQL + the alert query + the comparison.

In [0]:
# ── Solution Q17: pipeline-level vs data-level monitoring ──────────────────────
print("── PIPELINE-LEVEL MONITORING (is the pipeline healthy?) ──")
print("  Metrics: job status (success/fail), duration, retry count, DBU usage, executor count.")
print("  Source: system.lakeflow.job_runs, system.lakeflow.job_tasks.")
print("  Question answered: 'Did the pipeline run and finish without errors?'")
print("  GAP: the pipeline can SUCCEED but produce WRONG data (e.g., upstream source changed schema,")
print("    row count dropped 50% but no error, nulls in a critical column).")
print()
print("── DATA-LEVEL MONITORING (is the data healthy?) ──")
print("  Metrics:")
print("    FRESHNESS: time since last data update (e.g., 'last row was 2 hours ago — SLA is 1 hour').")
print("    VOLUME: row count vs 7-day average (e.g., 'today has 500K rows, average is 600K — 17% drop').")
print("    QUALITY: null rate on critical columns, distinct count, schema match, DQ pass rate.")
print("    COMPLETENESS: all expected partitions present (e.g., 'missing hour 14 partition').")
print("  Question answered: 'Is the data correct, complete, and fresh?'")
print()

print("── METRICS-TABLE PATTERN ──")
metrics_ddl = """
CREATE TABLE workspace.monitoring.pipeline_metrics (
    pipeline_name    STRING,
    run_ts           TIMESTAMP,
    row_count        BIGINT,
    null_rate_pct    DOUBLE,
    freshness_min    INT,
    dq_pass_rate     DOUBLE,
    status           STRING
) USING DELTA;
"""
print(metrics_ddl)

insert_sql = """
-- After each pipeline run, insert a metrics row
INSERT INTO workspace.monitoring.pipeline_metrics
SELECT
    'silver_orders'                          AS pipeline_name,
    current_timestamp()                      AS run_ts,
    COUNT(*)                                  AS row_count,
    ROUND(SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS null_rate_pct,
    DATEDIFF(MINUTE, MAX(order_date), current_timestamp()) AS freshness_min,
    0.98                                     AS dq_pass_rate,  -- from DQ framework
    'SUCCESS'                                AS status
FROM workspace.default.silver_orders;
"""
print(insert_sql)

alert_sql = """
-- SQL ALERT: row count dropped > 20% from 7-day average
WITH baseline AS (
    SELECT AVG(row_count) AS avg_count
    FROM workspace.monitoring.pipeline_metrics
    WHERE pipeline_name = 'silver_orders'
      AND run_ts >= DATE_SUB(current_timestamp(), 7)
),
latest AS (
    SELECT row_count AS latest_count
    FROM workspace.monitoring.pipeline_metrics
    WHERE pipeline_name = 'silver_orders'
    ORDER BY run_ts DESC LIMIT 1
)
SELECT
    CASE WHEN latest_count < avg_count * 0.8 THEN 'ALERT: row count dropped > 20%'
         ELSE 'OK'
    END AS alert_status,
    latest_count,
    ROUND(avg_count, 0) AS avg_7d
FROM baseline, latest;
"""
print(alert_sql)

print("── LAKEHOUSE MONITORING (Databricks native) ──")
print("  Instead of building the metrics table manually, Databricks offers Lakehouse Monitoring:")
print("    CREATE MONITOR silver_orders
      AS (SELECT * FROM workspace.default.silver_orders)
      WITH BASE TABLE (SELECT * FROM workspace.default.silver_orders WHERE run_date < '2026-01-01')
      SCHEDULE EVERY 1 HOUR;")
print()
print("  Auto-generates: data drift (schema/value), freshness, volume, and custom SQL metrics.")
print("  Stores results in system monitoring tables — queryable with SQL.")
print()
print("── MANUAL METRICS TABLE vs LAKEHOUSE MONITORING ──")
print("  Manual: full control over what to track, works on any platform, but you maintain the code.")
print("  Lakehouse: auto-generated, no code, but limited to Databricks and predefined metrics.")
print("  Rule: use Lakehouse Monitoring for standard metrics; add a manual metrics table for custom KPIs.")
print()
print("Q17 OK — Pipeline vs data monitoring + metrics table + alert pattern ✓")

## Q18 [ADVANCED] Monitoring — Logging Best Practices: log4j, Python logging, Driver Logs

**Scenario**: A Spark job failed at 2 AM. The on-call engineer opens the Spark UI but finds only `print()` statements — unstructured, no levels, impossible to search. The next morning, the team lead says: "We need proper logging across all pipelines." This tests your knowledge of logging frameworks, what to log, and where to find logs in a Spark environment — asked at every senior data engineering interview.

**Requirements**:
1. Explain **log4j levels** and their hierarchy: ERROR > WARN > INFO > DEBUG > TRACE. Which level for which message?
2. Compare **Python `logging` module** vs `print()`: why logging wins (levels, structured fields, log delivery capture, filtering).
3. Show a logging setup for PySpark: create a logger, set the level, add structured fields (pipeline_name, batch_id, row_count).
4. Explain **where to find logs**: driver stdout/stderr in Spark UI, cluster log delivery to cloud storage, and how to search them.
5. Explain **what to log** at each pipeline stage: start (batch ID, source), progress (rows processed, partition), end (row count, duration, DQ result), error (stack trace, input that caused it).
6. Explain the **driver vs executor** log distinction: driver logs show orchestration; executor logs show task-level details.

**Expected output**: a logging code example + the what-to-log checklist + the log location guide.

In [0]:
# ── Solution Q18: logging best practices for Spark pipelines ───────────────────
import logging
import json
from datetime import datetime

# ── 1. LOG4J LEVELS ──
print("── LOG4J LEVELS (hierarchy: ERROR > WARN > INFO > DEBUG > TRACE) ──")
print("  ERROR: Something failed, needs human attention. (e.g., 'Batch 42 failed: schema mismatch')")
print("  WARN:  Unexpected but recoverable. (e.g., 'Skipped 3 rows with null key, continuing')")
print("  INFO:  Normal progress. (e.g., 'Batch 42: 1.2M rows processed in 45s')")
print("  DEBUG: Detailed internals. (e.g., 'Partition 7: 150K rows, 200MB shuffled')")
print("  TRACE: Very fine-grained. (e.g., 'Reading file s3://bucket/path/file_001.parquet')")
print("  Production: set level to INFO. For debugging: set to DEBUG.")
print()

# ── 2. PYTHON logging vs print() ──
print("── PYTHON logging vs print() ──")
print("  print(): stdout only, no level, no structure, not captured by cluster log delivery.")
print("  logging: levels, structured fields, captured by cluster log delivery, searchable in UI.")
print("  RULE: NEVER use print() in production Spark code. Always use logging.")
print()

# ── 3. Logging setup for PySpark ──
logger = logging.getLogger("pipeline.silver_orders")
logger.setLevel(logging.INFO)

# Prevent duplicate handlers in Spark (driver may add multiple)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        '%(asctime)s [%(levelname)s] %(name)s - %(message)s'
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

# Structured logging with extra fields
batch_id = "batch_2026_01_17_003"
logger.info(f"Pipeline started",
            extra={"batch_id": batch_id, "pipeline": "silver_orders", "stage": "start"})
logger.info(f"Rows processed: 1200000",
            extra={"batch_id": batch_id, "pipeline": "silver_orders", "stage": "progress"})
logger.warning(f"3 rows with null customer_id skipped",
               extra={"batch_id": batch_id, "pipeline": "silver_orders", "null_count": 3})
logger.info(f"Pipeline completed: 1200000 rows, 45.2s, DQ pass rate 99.8%",
            extra={"batch_id": batch_id, "pipeline": "silver_orders", "stage": "end",
                   "row_count": 1200000, "duration_sec": 45.2, "dq_pass_rate": 99.8})

# ── 4. WHERE TO FIND LOGS ──
print()
print("── WHERE TO FIND LOGS IN SPARK ──")
print("  1. SPARK UI → 'Logs' tab on the cluster page → driver stdout/stderr.")
print("     Look for [ERROR] and [WARNING] lines. Search by timestamp or batch_id.")
print("  2. CLUSTER LOG DELIVERY: if configured, logs are delivered to S3/ADLS/GCS.")
print("     Search with: SELECT * FROM log_files WHERE content LIKE '%ERROR%' AND date = '2026-01-17'")
print("  3. DATABRICKS JOBS UI → failed run → 'Logs' tab → driver + executor logs.")
print("  4. system.compute.clusters / system.lakeflow.* for structured event logs.")
print()

# ── 5. WHAT TO LOG AT EACH STAGE ──
print("── WHAT TO LOG AT EACH PIPELINE STAGE ──")
print("  START:    batch_id, source table/path, expected row count (if known), start time.")
print("  PROGRESS: rows processed, partitions read, bytes scanned, shuffle bytes.")
print("  DQ CHECK: rule name, pass/fail, failed row count, sample of failed rows.")
print("  END:      total rows, duration, DQ pass rate, output table version.")
print("  ERROR:    exception type, message, stack trace, the input row/batch that triggered it.")
print()

# ── 6. DRIVER vs EXECUTOR LOGS ──
print("── DRIVER vs EXECUTOR LOGS ──")
print("  DRIVER: orchestration — job started, DAG stages, broadcast variables, accumulators.")
print("    Shows: 'Stage 3 started with 200 tasks', 'Job 0 finished in 45s'.")
print("    Use for: understanding overall pipeline flow and failures.")
print("  EXECUTOR: task-level — each task's work, local computation, spills.")
print("    Shows: 'Task 42 processing partition 7: 150K rows', 'Spilling sort to disk'.")
print("    Use for: diagnosing data skew, OOM on a specific executor, slow tasks.")
print()
print("  In Spark UI: Stages tab → click a stage → see task-level logs per executor.")
print("  For Python UDFs: executor logs show the Python worker's stdout/stderr.")
print()
print("Q18 OK — Logging best practices, levels, locations, and what-to-log checklist ✓")

## 🏁 Day 17 Score Card

Track your performance — be honest, the point is to find gaps *before* the real interview.

| Q | Topic | Tag | Attempted (✓/✗) | Confident (1-5) | Revisit? |
| --- | --- | --- | --- | --- | --- |
| Q1 | Recursive CTE: employee hierarchy with depth & path | ADVANCED | | | |
| Q2 | Time-series gap filling & forward fill | ADVANCED | | | |
| Q3 | Sliding window maximum (monotonic deque) | ADVANCED | | | |
| Q4 | Number of islands (BFS/DFS on a grid) | ADVANCED | | | |
| Q5 | CDC processing with MERGE INTO | ADVANCED | | | |
| Q6 | Schema evolution: auto-merge & column addition | ADVANCED | | | |
| Q7 | Tungsten whole-stage code generation | ADVANCED | | | |
| Q8 | JVM GC tuning for Spark executors | HARD | | | |
| Q9 | Dead letter queue for bad records | ADVANCED | | | |
| Q10 | Delta OPTIMIZE, ZORDER & Liquid Clustering | HARD | | | |
| Q11 | Idempotent writes & checkpoint recovery | ADVANCED | | | |
| Q12 | Data Vault 2.0: hubs, links, satellites | HARD | | | |
| Q13 | Bridge tables for many-to-many relationships | ADVANCED | | | |
| Q14 | Snowflake materialized views vs dynamic tables | ADVANCED | | | |
| Q15 | Resource monitors & warehouse sizing | ADVANCED | | | |
| Q16 | Airflow vs Databricks Jobs vs ADF | ADVANCED | | | |
| Q17 | Pipeline-level vs data-level monitoring | ADVANCED | | | |
| Q18 | Logging best practices: log4j, Python logging | ADVANCED | | | |

**Self-review prompts:**

1. Which question took the longest to articulate out loud? Practice it twice more tomorrow.
2. For every ✗: write the one-sentence core idea on a flashcard (e.g., *"recursive CTE: anchor + UNION ALL + join children to parent already in CTE"*).
3. Day 17 deepens Week 3: you should now be comfortable defending trade-offs for Tungsten codegen (volcano vs fused), Data Vault (agility vs query simplicity), and DLQ (fail-fast vs continue). If any are shaky, revisit Day 16's circuit breaker and exactly-once content.
4. Tomorrow: Day 18 (ADVANCED — Week 3 day 4, pushing further toward HARD with distributed caching, CBO internals, and streaming backpressure).